# 🚀 Chapter 6: Advanced Logistic Regression and Extensions
**Book Reference:** *scikit-learn Cookbook, Third Edition*

---
## 1. Introduction
Logistic Regression is more than just a basic binary classification algorithm. In practice, we often encounter datasets with multiple categories (*multiclass*), massive datasets requiring specialized optimization algorithms (*solvers*), or highly skewed data distributions (*imbalanced*). This chapter covers the advanced capabilities of `LogisticRegression` within scikit-learn.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV

%matplotlib inline
np.random.seed(42)

## 2. Multiclass Classification (OVR vs Multinomial)
By default, Logistic Regression predicts 2 classes (Binary). For more than 2 classes, scikit-learn utilizes two main strategies:
- **One-Versus-Rest (OvR):** Builds a separate binary model for each class, evaluating it against all remaining classes combined.
- **Multinomial (Softmax):** Directly optimizes the cross-entropy loss function across all classes simultaneously. This approach is typically more accurate but requires higher computational resources.

In [2]:
# Generate a dataset with 3 classes (Multiclass)
X_multi, y_multi = make_blobs(n_samples=300, centers=3, n_features=2, random_state=42, cluster_std=2.0)
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_multi, y_multi, test_size=0.3)

# Model 1: One-Versus-Rest (OvR)
log_ovr = LogisticRegression(multi_class='ovr', solver='lbfgs')
log_ovr.fit(X_train_m, y_train_m)

# Model 2: Multinomial (Cross-Entropy)
log_multi = LogisticRegression(multi_class='multinomial', solver='lbfgs')
log_multi.fit(X_train_m, y_train_m)

print(f"OvR Accuracy: {log_ovr.score(X_test_m, y_test_m):.3f}")
print(f"Multinomial Accuracy: {log_multi.score(X_test_m, y_test_m):.3f}")

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


OvR Accuracy: 1.000
Multinomial Accuracy: 0.989


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


## 3. Tuning Solvers and Regularization (SAGA & Elastic-Net)
The `solver` parameter is highly critical. The default solver `lbfgs` only supports L2 (Ridge) regularization. If you are working with extremely large datasets or want to apply a blended L1 & L2 penalty (Elastic-Net), you should choose the **'saga'** solver.

In [3]:
# Deploy the 'saga' solver which supports an 'elasticnet' penalty
log_saga = LogisticRegression(solver='saga', penalty='elasticnet', l1_ratio=0.5, max_iter=5000)
log_saga.fit(X_train_m, y_train_m)

print("Model Coefficients (SAGA - ElasticNet):")
print(log_saga.coef_)
print(f"\nSAGA ElasticNet Accuracy: {log_saga.score(X_test_m, y_test_m):.3f}")

Model Coefficients (SAGA - ElasticNet):
[[-0.28515879  1.11644956]
 [ 1.27908687  0.        ]
 [-0.02428077 -0.65188982]]

SAGA ElasticNet Accuracy: 1.000


## 4. LogisticRegressionCV (Built-in Cross-Validation)
Similar to `RidgeCV` or `LassoCV`, scikit-learn includes the `LogisticRegressionCV` class. It efficiently scans for the optimal regularization strength hyperparameter (`C`, which is the inverse of `alpha`) automatically without needing the slower `GridSearchCV` class.

In [4]:
from sklearn.linear_model import LogisticRegressionCV

# Search for the best C hyperparameter automatically using 5-fold cross-validation
log_cv = LogisticRegressionCV(cv=5, random_state=42, max_iter=1000)
log_cv.fit(X_train_m, y_train_m)

print("Best C values (Inverse Regularization Strength) discovered for each class:")
print(log_cv.C_)
print(f"\nLogisticRegressionCV Accuracy: {log_cv.score(X_test_m, y_test_m):.3f}")

Best C values (Inverse Regularization Strength) discovered for each class:
[0.0001 0.0001 0.0001]

LogisticRegressionCV Accuracy: 1.000


## 5. Handling Imbalanced Data with `class_weight`
In real-world applications (such as fraud detection), the positive class of interest might represent only 1% of the total dataset. A standard model will likely always predict the majority class ("Not Fraud") to claim a misleading 99% accuracy rate, which provides no analytical value.

Configuring the parameter `class_weight='balanced'` instructs the algorithm to apply a much higher penalty cost to classification errors made on the minority class.

In [5]:
# Construct a heavily imbalanced dataset (95% Class 0, 5% Class 1)
X_imb, y_imb = make_classification(n_samples=1000, n_classes=2, weights=[0.95, 0.05],
                                   random_state=42, n_clusters_per_class=1)
Xi_train, Xi_test, yi_train, yi_test = train_test_split(X_imb, y_imb, test_size=0.3, random_state=42)

# 1. Standard Model (Biased toward the majority class)
log_standard = LogisticRegression()
log_standard.fit(Xi_train, yi_train)

# 2. Balanced Model (Assigns higher significance weight to the minority class)
log_balanced = LogisticRegression(class_weight='balanced')
log_balanced.fit(Xi_train, yi_train)

print("=== Recall Evaluation Comparison (Detecting Minority Class 1) ===\n")
print("1. Standard Logistic Regression:")
print(classification_report(yi_test, log_standard.predict(Xi_test)))

print("\n2. Balanced Logistic Regression:")
print(classification_report(yi_test, log_balanced.predict(Xi_test)))

print("*(Observe how the Recall metric for class 1 increases sharply when using the Balanced model configurations)*")

=== Recall Evaluation Comparison (Detecting Minority Class 1) ===

1. Standard Logistic Regression:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       283
           1       1.00      0.59      0.74        17

    accuracy                           0.98       300
   macro avg       0.99      0.79      0.86       300
weighted avg       0.98      0.98      0.97       300


2. Balanced Logistic Regression:
              precision    recall  f1-score   support

           0       0.99      0.94      0.96       283
           1       0.42      0.76      0.54        17

    accuracy                           0.93       300
   macro avg       0.70      0.85      0.75       300
weighted avg       0.95      0.93      0.94       300

*(Observe how the Recall metric for class 1 increases sharply when using the Balanced model configurations)*
